# A-ADP Colab Training

**Before you start:**
1. Runtime → Change runtime type → **A100 GPU**
2. Add two Colab Secrets (🔑 icon in left sidebar):
   - `HF_TOKEN` — your HuggingFace token with CT-RATE access
   - `WANDB_API_KEY` — your Weights & Biases API key (optional; leave blank to skip)
3. Mount Google Drive before training — checkpoints are saved there so a Colab disconnect does not lose progress.

**Run order:** cells are numbered. Run 0 → 1 → 2 → … in sequence.  
After a disconnect, re-run cells 0–3 only (setup), then skip to whichever training cell you were on and use `--resume`.

In [ ]:
# Cell 0 — Verify A100
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout)
assert 'A100' in result.stdout or 'L4' in result.stdout, (
    'Not on A100/L4 — change runtime type to GPU (A100) before proceeding.'
)

In [ ]:
# Cell 1 — Mount Google Drive (checkpoints persist across sessions)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'   # pre-downloaded NIfTI files

os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
print('Drive mounted.  Checkpoints →', DRIVE_DIR)
print('CT-RATE cache  →', LOCAL_DATA_DIR)

In [ ]:
# Cell 2 — Get the codebase
#
# OPTION A (recommended): clone from a private GitHub repo
#   Replace YOUR_GH_TOKEN and YOUR_USERNAME/REPO with your actual values.
#   Push the repo from your local machine first:
#       git remote add origin https://github.com/YOUR_USERNAME/REPO.git
#       git push -u origin main
#
# Uncomment and fill in:
# GH_TOKEN = 'ghp_...'   # fine-grained PAT with repo read scope
# !git clone https://{GH_TOKEN}@github.com/YOUR_USERNAME/3d-compression.git /content/3d-compression

# OPTION B: zip upload
#   On your local machine:  git archive HEAD -o aadp.zip
#   Upload aadp.zip to /content/ via the file browser, then:
# !unzip -q /content/aadp.zip -d /content/3d-compression

%cd /content/3d-compression
!ls

In [ ]:
# Cell 3 — Install dependencies (~4 min on first run)
!pip install -q -r requirements.txt

In [ ]:
# Cell 4 — Load secrets into environment
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

try:
    wb_key = userdata.get('WANDB_API_KEY')
    if wb_key:
        os.environ['WANDB_API_KEY'] = wb_key
        import wandb
        wandb.login(key=wb_key, relogin=True)
        print('W&B login OK')
    else:
        print('No WANDB_API_KEY — W&B disabled')
        os.environ['WANDB_MODE'] = 'disabled'
except Exception:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled')

print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# Cell 4b — Pre-download CT-RATE subset to Drive
#
# Run this ONCE per Drive account (files persist across Colab sessions).
# Re-running is safe — already-downloaded files are skipped automatically.
#
# With 5 GB free on Drive:
#   ~60 training + 10 val volumes ≈ 3.5 GB data  (50 MB avg per volume)
#   ~1.5 GB remaining for checkpoints + figures
#
# Adjust n_train / n_valid if your available space differs.

import os, shutil

DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'

# Show available Drive space before downloading
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
free_gb = free / 1e9
print(f'Drive free space: {free_gb:.1f} GB')

# Estimate how many volumes we can safely download (50 MB avg, keep 1 GB buffer)
max_volumes = max(0, int((free_gb - 1.0) * 1e9 / (50 * 1e6)))
n_train = min(60, max_volumes - 10)   # reserve 10 slots for val
n_valid = min(10, max_volumes - n_train)

print(f'Targeting {n_train} train + {n_valid} val volumes '
      f'(≈ {(n_train + n_valid) * 50 / 1024:.1f} GB estimated)')

if n_train <= 0:
    print('Not enough free space — free up Drive space and re-run.')
else:
    from aadp.data.ctrate_dataset import download_subset_to_disk

    download_subset_to_disk(
        local_data_dir=LOCAL_DATA_DIR,
        n_train=n_train,
        n_valid=n_valid,
        token=os.environ.get('HF_TOKEN'),
        max_workers=4,      # 4 parallel downloads; raise to 8 on fast connections
    )

    total2, used2, free2 = shutil.disk_usage('/content/drive/MyDrive')
    print(f'\nDrive free after download: {free2 / 1e9:.1f} GB')
    print(f'Space used by cache: {(used2 - used) / 1e9:.1f} GB')

---
## Step 0 — 50-sample smoke test
Runs one epoch on 50 training samples and 10 val samples.  
Expected: loss decreases, checkpoint saved to Drive, no errors.  
Expected time: ~3 minutes on A100.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'

!python scripts/train.py \
    --config configs/aadp_base.yaml \
    --max_samples 50 \
    --set num_epochs=1 \
          use_wandb=false \
          checkpoint_dir={DRIVE_DIR}/checkpoints/smoke_test \
          val_every_n_steps=10 \
          save_every_n_steps=25 \
          local_data_dir={LOCAL_DATA_DIR}

---
## Step 1 — Train A-ADP base

**Phase 1: Quick validation run** (cells 6a)  
3 epochs × 2,000 samples ≈ 2–3 h on A100. Gets you first VTCB numbers fast.

**Phase 2: Full run** (cell 6b)  
10 epochs, all data. Use `--resume` after any disconnect.  
Checkpoints saved every 500 steps to Drive, best checkpoint also saved.

Start with 6a. If the loss trend looks right (dropping, no NaN), run 6b.

In [ ]:
# Cell 6a — A-ADP quick validation (3 epochs, local cache)
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'

!python scripts/train.py \
    --config configs/aadp_base.yaml \
    --set num_epochs=3 \
          checkpoint_dir={DRIVE_DIR}/checkpoints/aadp_base \
          local_data_dir={LOCAL_DATA_DIR}

In [ ]:
# Cell 6b — A-ADP full training (10 epochs, local cache)
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'

!python scripts/train.py \
    --config configs/aadp_base.yaml \
    --set checkpoint_dir={DRIVE_DIR}/checkpoints/aadp_base \
          local_data_dir={LOCAL_DATA_DIR}

# To resume after disconnect (uncomment):
# !python scripts/train.py \
#     --config configs/aadp_base.yaml \
#     --resume {DRIVE_DIR}/checkpoints/aadp_base/checkpoint_best.pt \
#     --set checkpoint_dir={DRIVE_DIR}/checkpoints/aadp_base \
#           local_data_dir={LOCAL_DATA_DIR}

In [ ]:
# Cell 7 — Perceiver baseline (local cache)
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'

!python scripts/train.py \
    --config configs/baseline_perceiver.yaml \
    --set checkpoint_dir={DRIVE_DIR}/checkpoints/perceiver \
          local_data_dir={LOCAL_DATA_DIR}

# Resume:
# !python scripts/train.py --config configs/baseline_perceiver.yaml \
#     --resume {DRIVE_DIR}/checkpoints/perceiver/checkpoint_best.pt \
#     --set checkpoint_dir={DRIVE_DIR}/checkpoints/perceiver \
#           local_data_dir={LOCAL_DATA_DIR}

In [ ]:
# Cell 8 — MedPruner baseline (local cache)
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'

!python scripts/train.py \
    --config configs/baseline_medpruner.yaml \
    --set checkpoint_dir={DRIVE_DIR}/checkpoints/medpruner \
          local_data_dir={LOCAL_DATA_DIR}

# Resume:
# !python scripts/train.py --config configs/baseline_medpruner.yaml \
#     --resume {DRIVE_DIR}/checkpoints/medpruner/checkpoint_best.pt \
#     --set checkpoint_dir={DRIVE_DIR}/checkpoints/medpruner \
#           local_data_dir={LOCAL_DATA_DIR}

---
## Step 2 — VTCB Evaluation

Sweeps M ∈ {16, 32, 64, 128} on the validation set for each trained model.  
Results saved as JSON files to Drive.  
Expected time: ~20 min per model on A100 (RadGraph DyGIE cold start adds ~2 min first time).

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'

for model_name, config, ckpt in [
    ('aadp',      'configs/aadp_base.yaml',          f'{DRIVE_DIR}/checkpoints/aadp_base/checkpoint_best.pt'),
    ('perceiver', 'configs/baseline_perceiver.yaml',  f'{DRIVE_DIR}/checkpoints/perceiver/checkpoint_best.pt'),
    ('medpruner', 'configs/baseline_medpruner.yaml',  f'{DRIVE_DIR}/checkpoints/medpruner/checkpoint_best.pt'),
]:
    print(f'\n=== Evaluating {model_name} ===')
    !python scripts/evaluate.py \
        --config {config} \
        --checkpoint {ckpt} \
        --budgets 16 32 64 128 \
        --results_dir {DRIVE_DIR}/results \
        --model_name {model_name}

---
## Step 3 — Validate the FiLM hypothesis: etext_matters after training

This is the empirical validation of A-ADP's core claim.  
At init, FiLM is identity so `etext_matters = False`.  
After training, FiLM weights should have shifted so different instructions produce different attention maps → `etext_matters = True`.

If `etext_matters` is still `False` after training, FiLM is not learning — that's the signal to activate `use_attn_loss: true`.

In [ ]:
import sys, os
sys.path.insert(0, '/content/3d-compression')
import torch

DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
device = torch.device('cuda')

from aadp.models.vlm import MedVLM
from aadp.training.factory import build_projector
import yaml

config = yaml.safe_load(open('configs/aadp_base.yaml'))

model = MedVLM(
    vit_model_name=config['vit_model_name'],
    vit_pretrained=True,
    vit_frozen=True,
    llm_model_name=config['llm_model_name'],
    llm_frozen=True,
    num_latents=config.get('num_latents', 32),
    num_tokens=config.get('num_tokens', 64),
    device=device,
).to(device)

ckpt_path = f'{DRIVE_DIR}/checkpoints/aadp_base/checkpoint_best.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.projector.load_state_dict(ckpt['projector'])
model.eval()

torch.manual_seed(0)
B, D, N, C, M = 2, 16, 196, 768, 64
patch_tokens = torch.randn(B, D, N, C, device=device)
etext_a = torch.randn(B, 768, device=device)
etext_b = torch.randn(B, 768, device=device)

with torch.no_grad():
    out_a = model.projector(patch_tokens, etext_a, H_patches=14, W_patches=14)
    out_b = model.projector(patch_tokens, etext_b, H_patches=14, W_patches=14)

etext_matters = not torch.allclose(out_a, out_b)
print(f'Output shape: {out_a.shape}')
print(f'etext_matters (after training): {etext_matters}')

if etext_matters:
    print('✓  FiLM is learning — A-ADP hypothesis confirmed.')
else:
    print('✗  FiLM not differentiating outputs.  See note below.')
    print('   → Re-train with use_attn_loss: true in configs/aadp_ablation_aux_loss.yaml')

---
## Step 4 — Attention map figures

Pull 2–3 representative volumes from the val set and generate `compare_instructions()` figures.  
Use contrasting queries (nodule vs pleural effusion, chest vs abdomen) to show steering.

These are the key qualitative figures for the paper.

In [ ]:
import sys, os, torch
sys.path.insert(0, '/content/3d-compression')

DRIVE_DIR = '/content/drive/MyDrive/aadp_training'
LOCAL_DATA_DIR = f'{DRIVE_DIR}/ct_rate_cache'
FIG_DIR = f'{DRIVE_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

from aadp.models.vlm import MedVLM
from aadp.data.ctrate_dataset import CTRATEDataset
from visualization.attention_maps import visualize_attention, compare_instructions
import yaml

device = torch.device('cuda')
config = yaml.safe_load(open('configs/aadp_base.yaml'))

model = MedVLM(
    vit_model_name=config['vit_model_name'],
    vit_pretrained=True,
    vit_frozen=True,
    llm_model_name=config['llm_model_name'],
    llm_frozen=True,
    num_latents=config.get('num_latents', 32),
    num_tokens=config.get('num_tokens', 64),
    device=device,
).to(device)

ckpt = torch.load(f'{DRIVE_DIR}/checkpoints/aadp_base/checkpoint_best.pt',
                  map_location=device, weights_only=False)
model.projector.load_state_dict(ckpt['projector'])
model.eval()

# Sample 3 volumes from val set (loads from local cache if available)
val_ds = CTRATEDataset(split='valid', max_samples=3, shuffle=False,
                       token=os.environ.get('HF_TOKEN'),
                       local_data_dir=LOCAL_DATA_DIR)
volumes = [item[0] for item in val_ds]

QUERY_PAIRS = [
    ('Are there any lung nodules?', 'Is there pleural effusion?'),
    ('Describe findings in the mediastinum.', 'Any evidence of pneumothorax?'),
]

for vi, vol in enumerate(volumes):
    for qi, (instr_a, instr_b) in enumerate(QUERY_PAIRS):
        compare_instructions(
            volume=vol,
            instructions=[instr_a, instr_b],
            checkpoints=[(model, 'A-ADP')],
            save_path=f'{FIG_DIR}/compare_vol{vi}_pair{qi}.png',
        )

for vi, vol in enumerate(volumes):
    with torch.no_grad():
        D, H, W = vol.shape
        slices = vol.unsqueeze(1).to(device)
        pts = model.vit(slices).float().unsqueeze(0)
        import math
        N = pts.shape[2]
        Hp = Wp = int(math.isqrt(N))
        etext = model.instruction_encoder(['Are there any lung nodules?']).float()
        model.projector(pts, etext, Hp, Wp)
    attn = model.projector.get_slice_attention()[0].cpu()
    visualize_attention(
        volume=vol,
        attn_weights=attn,
        instruction='Are there any lung nodules?',
        save_path=f'{FIG_DIR}/attn_vol{vi}.png',
    )

print('Figures saved to', FIG_DIR)
!ls {FIG_DIR}

---
## Step 5 — Compression–quality curves

In [ ]:
import sys
sys.path.insert(0, '/content/3d-compression')

DRIVE_DIR = '/content/drive/MyDrive/aadp_training'

from visualization.compression_curves import plot_paper_figures

plot_paper_figures(
    results_dir=f'{DRIVE_DIR}/results',
    save_dir=f'{DRIVE_DIR}/figures/compression_curves',
)

print('Saved:')
import os
for f in sorted(os.listdir(f'{DRIVE_DIR}/figures/compression_curves')):
    print(' ', f)

---
## Step 6 — Numeric comparison table

In [ ]:
import sys, json
sys.path.insert(0, '/content/3d-compression')

DRIVE_DIR = '/content/drive/MyDrive/aadp_training'

from aadp.evaluation.benchmarks.vtcb import VTCBRunner

result_jsons = {
    'A-ADP':      f'{DRIVE_DIR}/results/aadp_vtcb.json',
    'Perceiver':  f'{DRIVE_DIR}/results/perceiver_vtcb.json',
    'MedPruner':  f'{DRIVE_DIR}/results/medpruner_vtcb.json',
}

comparison = VTCBRunner.compare(result_jsons)

# Pretty-print as a table
metrics = sorted(comparison.keys())
models  = sorted({m for v in comparison.values() for m in v})
header  = f"{'Metric':<25}" + ''.join(f"{m:>12}" for m in models)
print(header)
print('-' * len(header))
for metric in metrics:
    row = f"{metric:<25}"
    for m in models:
        v = comparison[metric].get(m, float('nan'))
        row += f"{v:>12.4f}"
    print(row)

---
## Step 7 (conditional) — Aux attention loss

**Only run this if Step 3 showed `etext_matters = False` after training,  
or if the attention map figures show no differentiation between queries.**

This activates the auxiliary attention-alignment loss defined in `ablations/auxiliary_attention_loss.py`.  
The loss encourages the model to concentrate attention on query-relevant slices during training,  
giving FiLM a training signal beyond the report-generation objective alone.

In [ ]:
# Cell 14 — A-ADP + aux attention loss (only if Step 3 etext_matters == False)
DRIVE_DIR = '/content/drive/MyDrive/aadp_training'

!python scripts/train.py \
    --config configs/aadp_ablation_aux_loss.yaml \
    --set checkpoint_dir={DRIVE_DIR}/checkpoints/aadp_aux_loss \
          use_attn_loss=true

---
## Notes

### Recovering from a Colab disconnect
1. Re-run cells 0–4 (GPU check, Drive mount, `cd`, install, secrets).
2. On the training cell you were on, uncomment the `--resume` line and fill in the latest checkpoint step from Drive.
3. The trainer resumes from the exact step — no loss of progress.

### Expected training times on A100
| Mode | Time |
|------|------|
| Smoke test (50 samples, 1 epoch) | ~3 min |
| Quick validation (2k samples, 3 epochs) | ~2–3 h per model |
| Full training (all data, 10 epochs) | ~15–20 h per model |

Use quick validation mode first to verify loss is decreasing, then kick off the full run.

### W&B dashboard
If W&B is enabled, training curves appear at **wandb.ai → project: aadp**.  
Each experiment name matches the `experiment_name` field in the YAML config.

### The core empirical question
After training, run cell 10 (etext_matters check).  
- `True` → FiLM has learned to steer attention. The qualitative figures in cell 11 should show distinct attention patterns per query.  
- `False` → FiLM weights did not shift enough. Activate `use_attn_loss: true` and retrain (cell 14).